# Chapter 18: HHL Algorithm

The Harrow-Hassidim-Lloyd (HHL) algorithm solves a linear system $A\mathbf{x}=\mathbf{b}$ on a quantum computer, returning the normalized solution direction $|u\rangle=|x/\|x\|\rangle$. In theory it runs in $O(\log N)$ time versus $O(N)$ classically — an exponential speedup in system size $N$ (Book §18.1, Table 18.1). In practice, loading $b$, reading out $u$, condition-number growth, sparsity assumptions, and NISQ noise all conspire to prevent that advantage from materializing today; there are currently no robust HHL implementations for generic systems with $N>4$, and most are classical simulations.

We study HHL anyway because it proves that certain linear-algebra problems admit quantum speedup *in principle*, and because its building blocks — QPE, controlled rotation, and inverse QPE — recur throughout quantum computing. This notebook uses `Chapter18_HHL_functions.py` to run the hybrid HHL solver on small QLSPs and to study how the solution fidelity depends on the number of phase-estimation qubits $m$ and the number of measurement shots.

---

**Prerequisites:**
- See `Chapter02_QuantumSoftware.ipynb` for installation instructions

## Setup and imports

We import NumPy, Matplotlib, the Qiskit Aer simulator, and the `myHHL` class that encapsulates the full HHL pipeline (QPE, controlled rotation, inverse QPE) from `Chapter18_HHL_functions.py`.

In [1]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from Chapter18_HHL_functions import myHHL #type: ignore
from qiskit_aer import Aer
print('Setup complete!')

Setup complete!


## HHL Examples  *(Book §18.2, §18.8)*

Each example fixes a symmetric matrix $A$, a unit-norm right-hand side $b$, an eigenvalue upper bound $\lambda$ (`lambdaUpper`), and the number of phase-estimation qubits $m$. HHL relies on the eigen-expansion of $A$: writing $b=\sum_k b_k v_k$, the solution is $x=\sum_k (1/\lambda_k)\,b_k v_k$, so the algorithm effectively applies $1/\lambda_k$ to each eigencomponent. The options range from diagonal $2\times2$ systems to a $4\times4$ Poisson matrix; select one by setting `example`.

In [2]:
example = 1
debug = False
nShots = 1000
if (example == 1):
	A = np.array([[1,0],[0,0.75]])
	v0 = np.array([0,1])
	v1 = np.array([1,0])
	D = [0.75,1]
	b = np.array([1/np.sqrt(2),1/np.sqrt(2)])
	lambdaUpper = 2
	m = 2
elif (example == 2):
	A = np.array([[2,-1],[-1,2]])
	v0 = np.array([1/np.sqrt(2),1/np.sqrt(2)])
	v1 = np.array([1/np.sqrt(2),-1/np.sqrt(2)])
	D = [1,3]
	b = np.array([-1/np.sqrt(2),1/np.sqrt(2)])
	lambdaUpper = 6
	m = 5
elif (example == 3):
	A = np.array([[1,0,0,-0.5],[0,1,0,0],[0,0,1,0],[-0.5,0,0,1]])
	v0 = np.array([1/np.sqrt(2),0,0,1/np.sqrt(2)])
	v1 = np.array([0,1,0,0])
	v2 = np.array([0,0,1,0])
	v3 = np.array([1/np.sqrt(2),0,0,-1/np.sqrt(2)])
	D = [0.5,1,1,1.5]
	#a = [1/np.sqrt(4),1/np.sqrt(4),1/np.sqrt(4),1/np.sqrt(4)]
	a = [0,0,1,0]
	b = a[0]*v0 + a[1]*v1 + a[2]*v2 + a[3]*v3
	lambdaUpper = 3
	m = 2
	# Example for N-dimensional Poisson matrix
elif example == 4:
	N = 4  # dimension of the Poisson matrix
	A = 2 * np.eye(N) - np.eye(N, k=1) - np.eye(N, k=-1)
	# Dirichlet boundary conditions: b = [1, 0, ..., 0]
	b = np.zeros(N)
	b[0] = 1
	D = np.linalg.eigvalsh(A)
	lambdaUpper = 1.1 * np.max(D)  # margin so all eigenphases < 1
	m = 3

print('A:\n', A)
print('b:\n', b)

A:
 [[1.   0.  ]
 [0.   0.75]]
b:
 [0.70710678 0.70710678]


## HHL Execution on a specific QLSP  *(Book §18.8, Example 18.2, Listing 18.1)*

We instantiate `myHHL(A, b, lambdaUpper, m, nShots)` and call `executeHHL()` to run the circuit and recover the quantum solution `uHHL`. Comparing against the exact solution `uExact`, the fidelity $f=|\langle u_{\text{HHL}}|u_{\text{exact}}\rangle|^2$ measures agreement. For $A=\mathrm{diag}(1,0.75)$, $b=(1,1)/\sqrt2$, $\lambda=2$, $m=2$, the book reports `uHHL` $\approx[0.611,0.792]$ against `uExact` $=[0.6,0.8]$. Because measurement is phase-agnostic, $|u\rangle$ and $-|u\rangle$ yield identical statistics.

In [3]:
HHL = myHHL(A,b,lambdaUpper = lambdaUpper,
				 m = m, nShots = nShots)
	
# Execute main code
if (HHL.executeHHL()):
	print("uHHL: \t\t\t", HHL.uHHL)
	HHL.solveuExact()
	print('uExact: \t\t', HHL.uExact)
	fidelity = np.abs(np.dot(HHL.uHHL,HHL.uExact))**2
	print('fidelity:', fidelity)
	HHL.debugHHL()

uHHL: 			 [0.81799612 0.57522374]
uExact: 		 [0.6 0.8]
fidelity: 0.9043566142807387
Exact eigen values of A:
 [1.   0.75]
Exact eigen vectors of A:
 [[1. 0.]
 [0. 1.]]
Exact eigenphases of A:
 [0.35355339 0.26516504]
xExact: [0.70710678 0.94280904]


## HHL fidelity vs nShots for varying qubits  *(Book §18.8, Examples 18.3–18.5)*

Solution quality is quantified by the state fidelity $f_{qf}=|\langle\phi|\psi\rangle|^2$ (Eq. 18.23), which improves with two knobs: the number of qubits $m$ used to approximate the eigenphases, and the number of shots `nShots` used to sample the circuit. Since the results are statistical, we average the fidelity over 25 trials for each $(m,\text{nShots})$ pair and plot it against a logarithmic shots axis, reproducing Figures 18.8–18.10. Populate `mValues` (e.g. `[2,3,5]`) to run the sweep; more ill-conditioned systems need larger $m$ to capture the eigenphases.

In [ ]:
from matplotlib.ticker import FormatStrFormatter

fig, ax = plt.subplots()
shots = [10,25,50,100,500,1000]
ax.yaxis.set_major_formatter(FormatStrFormatter('%.4f'))
mValues = [2,3]
lineType = ['-','--',':','-.']
expt = 0
for m in mValues:
	fidelityResult= []
	print("m: ",m)
	for nShots in shots:
		print("nShots: ",nShots)
		fAverage = 0
		HHL = myHHL(A,b,lambdaUpper = lambdaUpper, m = m, nShots = nShots)
		HHL.solveuExact()
		nSuccessfulTrials = 0
		nTrials = 25
		for t in range(nTrials):
			if (not HHL.executeHHL()):
				continue
			nSuccessfulTrials = nSuccessfulTrials +1
			fidelity = np.abs(np.dot(HHL.uHHL,HHL.uExact))**2
			fAverage = fAverage + fidelity
		if nSuccessfulTrials > 0:
			fAverage = fAverage/nSuccessfulTrials
		else:
			fAverage = float('nan')  # all trials failed post-selection
			print('Warning: no successful trials for m=%d, nShots=%d' % (m, nShots))
		print("fAverage:", fAverage)
		fidelityResult.append(fAverage)
	
	print(fidelityResult)
	plt.semilogx(shots, fidelityResult,lineType[expt])
	expt = expt +1


plt.legend(['m=2','m=3','m=5'])
plt.grid(visible = True)
plt.xlabel("nShots", fontsize = 14)
plt.ylabel("Average fidelity", fontsize = 14)



m:  2
nShots:  10


## An engineering example: the 1D Poisson system  *(Book §18.9)*

The examples above were toy systems chosen to expose HHL's behavior. We now apply HHL to a genuine discretized differential operator: the 1D Poisson (steady-state diffusion) equation $-u''(x)=f(x)$, which under a central finite difference becomes $\mathbf{K}\mathbf{d}=\mathbf{f}$ with the tridiagonal matrix $\mathbf{K}$ having $2$ on the diagonal and $-1$ on the off-diagonals.

Two structural properties matter for HHL. First, $\mathbf{K}$ is **sparse** ($s=O(1)$: at most three nonzeros per row) — the regime HHL needs. Second, its eigenvalues have the closed form $\lambda_k = 4\sin^2\!\big(\tfrac{k\pi}{2(N+1)}\big)$, $k=1,\dots,N$, so the condition number grows as $\kappa=\lambda_{\max}/\lambda_{\min}=O(N^2)$. This is the crux: refining the mesh (larger $N$) makes the system progressively **more ill-conditioned**, and HHL's cost and error both scale with $\kappa$. Poisson is thus exactly the boundary case — sparse enough to be HHL-friendly, but conditioned in a way that punishes refinement.

In [ ]:
# 1D Poisson matrix and its condition-number growth (closed form)
def poisson(N):
    return 2*np.eye(N) - np.eye(N, k=1) - np.eye(N, k=-1)

print(f"{'N':>4}  {'lam_min':>9}  {'lam_max':>9}  {'kappa':>9}")
for N in [4, 8, 16, 32]:
    ev = np.linalg.eigvalsh(poisson(N))
    print(f"{N:>4}  {ev.min():>9.4f}  {ev.max():>9.4f}  {ev.max()/ev.min():>9.2f}")

### HHL on the $N=4$ Poisson system

We solve the $N=4$ case ($\kappa\approx 9.5$) with a unit source at the first node, $\mathbf{f}=(1,0,0,0)$. As with the toy examples we set $\overline\lambda$ just above the largest eigenvalue (a $1.1\times$ margin so no eigenphase wraps at $1$), and compare the recovered $\mathbf{u}_{\text{HHL}}$ against the classical solution by fidelity.

In [ ]:
N = 4
K = poisson(N)
f = np.zeros(N); f[0] = 1.0            # unit source at node 0
lambdaUpperP = 1.1 * np.max(np.linalg.eigvalsh(K))

HHLp = myHHL(K, f, lambdaUpper=lambdaUpperP, m=3, nShots=2000)
if HHLp.executeHHL():
    HHLp.solveuExact()
    fid = np.abs(np.dot(HHLp.uHHL, HHLp.uExact))**2
    print('uHHL:  ', np.round(HHLp.uHHL, 4))
    print('uExact:', np.round(HHLp.uExact, 4))
    print(f'fidelity: {fid:.4f}')

### Discussion

The Poisson fidelity (typically $\sim 0.7$–$0.8$) is markedly lower than the toy examples above, which reached $>0.95$. The reason is structural, not a bug: the smallest Poisson eigenvalue ($\lambda_{\min}\approx 0.38$ for $N=4$) is the one QPE resolves worst, and it sits close to the $\underline\lambda$ used in the controlled rotation, so small phase-estimation errors translate into large errors in $1/\lambda$. Increasing $m$ does **not** reliably help: more precision qubits deepen the circuit and give the eigenphase estimates more room to scatter, so fidelity can even decrease. This is the concrete face of the chapter's conditioning caveat — for a discretized PDE operator whose $\kappa$ grows as $O(N^2)$, HHL's accuracy degrades exactly where an engineer would want to refine the mesh. Poisson is a legitimate HHL target in the sense that it is sparse, but it also marks the boundary where the algorithm's conditioning dependence begins to bite.